## Project 3 ChIP-seq Report

### **Introduction**

&nbsp;&nbsp;&nbsp;<font size="3"> RUNX1 is a transcription factor essential for regulating gene expression, chromatin structure, and cell identity in breast epithelial cells. Dysregulation of RUNX1 has been linked to breast cancer progression. The original study investigated RUNX1’s role in transcriptional regulation and genome organization in MCF-7 breast cancer cells by mapping genome-wide RUNX1 binding sites and analyzing their association with gene expression and chromatin interactions. The authors conducted RUNX1 ChIP-seq with two biological replicates and matched INPUT controls, enabling identification of reproducible RUNX1-bound regions while accounting for background signals. The study used bioinformatic analyses, including read alignment, peak calling, reproducibility filtering, genomic annotation, and motif enrichment to interpret RUNX1’s regulatory function. By replicating these analyses in my workflow, I evaluated RUNX1 binding locations in the genome and assessed how these patterns may influence transcriptional regulation in breast cancer.</font>

### **Methods**
&nbsp;&nbsp;&nbsp;<font size="3">The purpose of this project was to analyze ChIP-seq data to identify reproducible RUNX1 binding sites and characterize their genomic features and biological functions. The dataset consisted of RUNX1 immunoprecipitation (IP) samples and matched INPUT controls, each with two biological replicates (IP_rep1, INPUT_rep1 and IP_rep2, INPUT_rep2). All analyses were performed using Nextflow v24.04.4 and used the Boston University Shared Computing Cluster (SCC). Workflows were executed using the -profile singularity,cluster configuration, ensuring each module ran within the correct containerized environment. The completed pipeline included read quality assessment, adapter trimming, alignment, sorting and indexing, coverage generation, replicate correlation analysis, peak calling, reproducibility filtering, blacklist removal, genomic annotation, motif discovery, computeMatrix profiling, and GO enrichment analysis.</font>

&nbsp;&nbsp;&nbsp;<font size="3">Input data consisted of single-end FASTQ files for both IP and INPUT samples. Initial quality control was performed using FastQC v0.12.1 (ghcr.io/bf528/fastqc:latest) on default parameters to evaluate per-base Phred scores, GC content, and adapter contamination. All QC outputs were aggregated with MultiQC v1.25 (ghcr.io/bf528/multiqc:latest) using default settings to produce a combined HTML report. Adapter trimming was carried out using Trimmomatic v0.39 (ghcr.io/bf528/trimmomatic:latest) with the provided TruSeq3 adapter file.</font>

&nbsp;&nbsp;&nbsp;<font size="3">Trimmed reads were aligned to the human hg38 genome (GRCh38 primary assembly) using Bowtie2 v2.5.1 (ghcr.io/bf528/bowtie2:latest). A Bowtie2 index was generated using default parameters. Reads were aligned in default sensitive mode, and the resulting SAM files were converted to BAM, sorted, and indexed using SAMtools v1.18 (ghcr.io/bf528/samtools:latest). Using Bowtie2’s default sensitive mode ensures more accurate read placement, which is important for ChIP-seq data, as reads recovered from variable regions increase the reliability of downstream peak calling. Alignment statistics, including total reads, mapped reads, and percent alignment, were generated with samtools flagstat.</font>

&nbsp;&nbsp;&nbsp;<font size="3">To generate genome-wide signal tracks, deepTools bamCoverage v3.5.2 (ghcr.io/bf528/deeptools:latest) converted sorted BAM files into bigWig coverage tracks normalized to RPKM. Replicate similarity was evaluated using deepTools multiBigwigSummary, followed by plotCorrelation with default parameters, which generated a Pearson correlation heatmap and scatterplots of the genome-wide RUNX1 signal. Peak calling was performed using HOMER v4.11 (ghcr.io/bf528/homer_samtools:latest). Tag directories were created from each BAM file using makeTagDirectory. Peaks were identified with findPeaks -style factor, using the matched INPUT tag directory as background. HOMER output included total peak counts, FDR thresholds, peak sizes, and enrichment statistics. Reproducible peaks shared between replicates (IP_rep1, IP_rep2) were obtained using bedtools intersect v2.31.1 (ghcr.io/bf528/bedtools:latest). ENCODE blacklist regions were removed using bedtools intersect -v, producing a final high-confidence set of RUNX1 peaks.</font>

&nbsp;&nbsp;&nbsp;<font size="3">Genomic annotation was performed using HOMER’s annotatePeaks.pl with the hg38 genome and GENCODE v45 GTF annotation to assign peaks to promoters, TSS regions, introns, exons, and intergenic regions. Motif enrichment was performed with HOMER’s findMotifsGenome.pl using default parameters to identify enriched transcription factor motifs within RUNX1-bound chromatin regions. RUNX1 signal profiles around genes were generated using deepTools computeMatrix with UCSC hg38 gene annotations, and visualized with plotProfile to display RUNX1 enrichment across genes. For functional enrichment analysis, gene symbols from the annotated peaks were extracted and deduplicated. Enrichment was performed in Python using GSEAPY v1.1.2 via the Enrichr API, using the GO Biological Process 2023 library and default parameters. Output tables included adjusted p-values, enrichment scores, and term ranks. Results were visualized as horizontal bar plots of −log10(adjusted p-value). Significant biological processes associated with RUNX1 binding included mesenchymal cell proliferation, glucocorticoid response, Notch signaling, and lipid-associated SREBP pathways.</font>

&nbsp;&nbsp;&nbsp;<font size="3">This ChIP-seq workflow established a reproducible, containerized pipeline using modular Nextflow processes. By integrating Bowtie2 alignment, HOMER peak calling, reproducibility filtering, blacklist removal, genomic annotation, motif discovery, and GO enrichment analysis, I identified RUNX1 binding sites and the biological pathways associated with RUNX1 regulatory activity.</font>

### **Quality Control Evaluation**
&nbsp;&nbsp;&nbsp;<font size="3">The MultiQC report integrated results from FastQC, Trimmomatic, HOMER, and samtools flagstat to evaluate the quality of the raw reads and post-alignment (File 1). Across samples, the total number of reads ranged from 29.5–30.1 million in the IP samples and one sample had a lower depth of 10.0–10.7 million in the INPUT samples, showing consistent sequencing depth within each group. FastQC reported a high mean Phred quality score (Q > 30) across all bases, indicating strong base-calling accuracy and minimal sequencing errors. The per-sequence GC content for all samples was within the expected range for hg38 (28–30%), and no abnormalities were flagged in the per-base N content or sequence quality modules. Importantly, FastQC did not report any adapter contamination or overrepresented sequences, which is consistent with the minimal trimming observed in the Trimmomatic summary. All FastQC modules passed standard quality thresholds, confirming that the raw reads could be used for further downstream analysis.</font>

&nbsp;&nbsp;&nbsp;<font size="3">Post-alignment metrics from samtools flagstat further support the high quality of the dataset (Table 1). The IP samples showed 92–95% total alignment rates, consistent with successful ChIP enrichment, while the INPUT controls showed lower mapping rates (~28–30%). This is typical of INPUT libraries, which contain mostly background chromatin rather than targeted transcription factor binding sites. Duplicate reads were removed during HOMER tag directory creation, resulting in near-zero duplication rates in the final alignment statistics. Overall, these results indicate that the ChIP-seq data are of high quality, with strong IP enrichment, clean INPUT controls, and no major technical artifacts. The experiment is well-suited for downstream analyses such as peak calling, reproducibility filtering, annotation, and motif enrichment without additional preprocessing or quality correction.</font>

**File 1: MultiQC Quality Control Report**  
[Click here to open the MultiQC report](multiqc_report.html)

In [1]:
import pandas as pd
from IPython.display import Markdown, display
df = pd.read_csv("samtools-flagstat-dp.tsv", sep="\t")
df

,Sample,Total Reads,Total Passed QC,Mapped,Duplicates,Paired in Sequencing,Properly Paired,Self and mate mapped,Singletons,Mate mapped to diff chr,Diff chr (mapQ >= 5)
INPUT_rep1,29.578193,29.578193,28.463242,0,0,0,0,0,0,0,NaN
INPUT_rep2,10.696005,10.696005,10.031987,0,0,0,0,0,0,0,NaN
IP_rep1,28.578787,28.578787,27.654087,0,0,0,0,0,0,0,NaN
IP_rep2,28.547028,28.547028,28.062474,0,0,0,0,0,0,0,NaN


**Table 1. Samtools flagstat summary table.** This table summarizes total reads, mapped reads, and mapping efficiency for all samples as reported by samtools flagstat.

### **Signal Coverage Plot**
&nbsp;&nbsp;&nbsp;<font size="3">The signal coverage plots for IP_rep1 and IP_rep2 in Figure 1, and Figure 2 display the average ChIP-seq signal across all annotated genes, centered on the transcription start site (TSS) and extending 2 kb upstream and downstream. Both replicates show a strong and narrow enrichment peak at the TSS, followed by a rapid decrease in signal across the gene body and toward the transcription end site (TES). This pattern is consistent across the two biological replicates, indicating that the enrichment profile is reproducible. The pronounced enrichment centered at the transcription start site (TSS) indicates that the immunoprecipitated factor primarily binds near promoter regions, a signal profile typical of transcription factors associated with transcription initiation or promoter accessibility. This promoter-proximal binding pattern is consistent with the anticipated regulatory function of the factor, as TSS enrichment is generally associated with active gene regulation. The consistent TSS peak and reproducible profiles across replicates collectively support the conclusion that this factor regulates gene promoters through binding near the TSS.</font>

<img src="/projectnb/bf528/students/monsivam/project-3-monsivam/Project_3_Figures/IP_rep1.png" width="700"/>

**Figure 1. RUNX1 ChIP-seq Signal Intensity Across Gene Bodies (IP_rep1).** This plot shows the average RUNX1 ChIP-seq signal across all annotated genes for IP_rep1, aligned at the transcription start site (TSS) and transcription end site (TES). A sharp enrichment peak at the TSS indicates strong promoter-proximal RUNX1 binding. The overall pattern reflects robust ChIP enrichment and high-quality signal for this replicate.

<img src="/projectnb/bf528/students/monsivam/project-3-monsivam/Project_3_Figures/IP_rep2.png" width="700"/>

**Figure 2. RUNX1 ChIP-seq Signal Intensity Across Gene Bodies (IP_rep2).** This plot displays the average RUNX1 ChIP-seq signal across all annotated genes for IP_rep2. As in IP_rep1, a pronounced peak at the TSS demonstrates strong RUNX1 binding near gene promoters. The close similarity in signal shape and magnitude between IP_rep1 and IP_rep2 highlights the reproducibility and consistency of the ChIP-seq experiment.

### **Motif Finding**
&nbsp;&nbsp;&nbsp;<font size="3">Using the HOMER findMotifsGenome.pl results (Table 2), the top enriched motifs were dominated by the RUNX family (RUNX1, RUNX2, RUNX-AML), which show extremely significant enrichment and appear in roughly 18–23% of all peak regions. This strong signal is expected because the ChIP experiment targets a RUNX-related factor, and the presence of multiple RUNX motifs across independent datasets further validates that the reproducible peaks correspond to true biological binding sites rather than noise. In addition to RUNX motifs, several AP-1/bZIP family motifs (Fosl2, Fra2, JunB, Jun-AP1) were also enriched. These transcription factors are known cofactors that frequently cooperate with RUNX proteins in gene regulation, particularly in differentiation and immune response pathways. These findings indicate that the factor of interest binds directly to canonical RUNX motifs and likely recruits AP-1 family proteins to shared regulatory elements. This motif combination aligns with established RUNX co-binding interactions, reinforcing the quality and specificity of the ChIP-seq peaks.</font>


In [3]:
import pandas as pd

motifs = pd.read_csv("/projectnb/bf528/students/monsivam/project-3-monsivam/motifs_test/knownResults.txt", sep="\t",header=0)
motifs


,Motif Name,Consensus,P-value,Log P-value,q-value (Benjamini),# of Target Sequences with Motif(of 8572),% of Target Sequences with Motif,# of Background Sequences with Motif(of 38670),% of Background Sequences with Motif
0,RUNX(Runt)/HPC7-Runx1-ChIP-Seq(GSE22178)/Homer,SAAACCACAG,0.000000e+00,-1671.0,0.0,1867.0,21.78%,1674.4,4.33%
1,RUNX1(Runt)/Jurkat-RUNX1-ChIP-Seq(GSE29180)/Homer,AAACCACARM,0.000000e+00,-1265.0,0.0,1947.0,22.72%,2373.7,6.14%
2,RUNX2(Runt)/PCa-RUNX2-ChIP-Seq(GSE33889)/Homer,NWAACCACADNN,0.000000e+00,-823.6,0.0,1466.0,17.10%,1961.3,5.07%
3,RUNX-AML(Runt)/CD4+-PolII-ChIP-Seq(Barski_et_a...,GCTGTGGTTW,0.000000e+00,-773.6,0.0,1351.0,15.76%,1772.9,4.58%
4,GRHL2(CP2)/HBE-GRHL2-ChIP-Seq(GSE46194)/Homer,AAACYKGTTWDACMRGTTTB,1.000000e-197,-455.8,0.0,629.0,7.34%,656.5,1.70%
...,...,...,...,...,...,...,...,...,...
1001,IDD4(C2H2)/col-IDD4-DAP-Seq(GSE60143)/Homer,TTTGTCTTTWTB,1.000000e+00,0.0,1.0,375.0,4.38%,3478.7,8.99%
1002,IDD5(C2H2)/colamp-IDD5-DAP-Seq(GSE60143)/Homer,TTTTGTCTTTTTBTK,1.000000e+00,0.0,1.0,318.0,3.71%,3068.6,7.93%
1003,REM16(ABI3VP1)/col-REM16-DAP-Seq(GSE60143)/Homer,DTTTTTSCCGSMAAA,1.000000e+00,0.0,1.0,0.0,0.00%,0.0,0.00%
1004,REM19(REM)/colamp-REM19-DAP-Seq(GSE60143)/Homer,AAAAAAAA,1.000000e+00,0.0,1.0,348.0,4.06%,2454.0,6.35%


**Table 2. Enriched Transcription Factor Motifs Identified by HOMER findMotifsGenome.** This table shows the top known motifs enriched within the RUNX1 ChIP-seq peaks. RUNX family motifs (RUNX1, RUNX2, RUNX-AML) are the most significantly enriched, consistent with the experiment targeting a RUNX-related factor. Several AP-1/bZIP motifs (Fosl2, Fra2, JunB) also appear, suggesting potential cooperative interactions at shared regulatory elements.

### **Overlap your ChIPseq results with the original RNAseq data**
&nbsp;&nbsp;&nbsp;<font size="3">I began by examining RUNX1 binding at the two key lncRNA genes highlighted in Figures 2D and 2E of the original publication: MALAT1 and NEAT1. In my recreated genome browser views (Figure 3 for MALAT1 and Figure 4 for NEAT1), I observed clear, reproducible enrichment of the RUNX1 ChIP-seq signal at both loci. RUNX1 peaks appeared consistently in both immunoprecipitation replicates and were largely absent from the input controls, confirming statistically significant RUNX1 occupancy at these genes in my dataset. These findings are consistent with the authors’ observations and indicate that both lncRNAs are direct RUNX1-bound regions in my analysis.</font>

&nbsp;&nbsp;&nbsp;<font size="3">Comparing my genomic tracks to the published figures, I found that the overall binding profiles were highly similar. Both MALAT1 and NEAT1 displayed discrete, well-defined RUNX1 peaks overlapping promoter-proximal or gene-body regions (Figures 3 and 4). Some differences in peak intensity and shape were evident, likely due to technical factors such as sequencing depth, genome build alignment, visualization scaling, and the specific peak-calling and normalization methods employed. The published figures also include RNA-seq tracks, whereas my recreations present only the ChIP-seq signal, which contributes to differences in visual complexity. Despite these minor discrepancies, the main conclusion remains: both loci exhibit strong and reproducible RUNX1 binding.</font>

&nbsp;&nbsp;&nbsp;<font size="3">I then integrated my ChIP-seq peaks with RNA-seq differential expression results to recreate the combined analysis shown in Figure 2F of the original publication. Using my reproducible peaks and the GEO RNA-seq dataset, I generated the corresponding bar plot (Figure 5). As in the original study, I found that a subset of both up-regulated and down-regulated genes contains nearby RUNX1 peaks, particularly within ±5 kb of the transcription start site and within ±20 kb of the gene body. However, the exact number of overlapping genes in my analysis did not match the values reported in the paper. This discrepancy is expected and can be attributed to several factors. First, the number and location of peaks in my filtered set may differ from those in the authors’ set due to variations in peak-calling thresholds, reproducibility filtering, and sequencing depth. Second, differences in RNA-seq processing, such as updated gene annotations, transcript-to-gene assignments, or differences in significance thresholds, can alter the counts of differentially expressed genes and their overlap with RUNX1 peaks. Additional variability may arise from technical sources such as library complexity or normalization strategies.</font>

&nbsp;&nbsp;&nbsp;<font size="3">Integrating ChIP-seq and RNA-seq analyses is essential for identifying direct RUNX1 target genes and distinguishing them from indirect transcriptional effects. ChIP-seq identifies RUNX1 binding sites, while RNA-seq reveals genes with altered expression. Overlaying these datasets enables more robust biological conclusions by assessing whether RUNX1 binding correlates with differential expression. This integrated approach highlights functional RUNX1 binding events and supports the interpretation that RUNX1 directly regulates a specific subset of genes. My recreated figures Figures 3,4,5 collectively reinforce the regulatory relationships described in the original study, despite expected numerical differences due to technical variation.</font>

<img src="/projectnb/bf528/students/monsivam/project-3-monsivam/Project_3_Figures/MALAT1_RUNX1_ChIP.png" width="700"/>

**Figure 3. RUNX1 ChIP-seq Signal Tracks at the MALAT1 locus across both IP and INPUT replicates (Recreation of Figure 2D).** This genome browser tracks display RUNX1 ChIP-seq coverage for both IP and INPUT replicates across the MALAT1 locus and its antisense transcript TALAM1. The IP tracks show strong, localized enrichment that is largely absent in the INPUT controls, indicating reproducible RUNX1 binding at this region. RefSeq gene annotations provide genomic context and highlight that RUNX1 occupancy occurs near regulatory elements associated with MALAT1.

<img src="/projectnb/bf528/students/monsivam/project-3-monsivam/Project_3_Figures/NEAT1_RUNX1_ChIP.png" width="700"/>

**Figure 4. RUNX1 ChIP-seq Signal Tracks at the NEAT1 Locus across both IP and INPUT replicates (Recreation of Figure 2E).** This genome browser view displays RUNX1 ChIP-seq signal across the NEAT1 locus for both biological replicates. Input controls (INPUT_rep1 and INPUT_rep2) are shown in light gray to represent background signal, while RUNX1 immunoprecipitated samples (IP_rep1 and IP_rep2) show enriched read coverage peaks corresponding to RUNX1 binding events. Gene models from RefSeq are shown at the bottom to provide genomic context, allowing visualization of binding relative to the NEAT1 promoter and transcript structure.

<img src="/projectnb/bf528/students/monsivam/project-3-monsivam/Project_3_Figures/RunX1 Binding vs RNA-seq differential expression.png" width="700"/>

**Figure 5. RUNX1 Binding Relative to Differentially Expressed Genes (Recreation of Figure 2F).** This bar plot shows the percentage of up- and down-regulated genes that overlap a RUNX1 ChIP-seq peak within ±5 kb of the TSS or ±20 kb of the gene. A subset of DE genes is RUNX1-bound, illustrating its contribution to transcriptional regulation.


### **Comparing key findings to the original paper**
&nbsp;&nbsp;&nbsp;<font size="3">To compare my results with the supplementary figures in the original publication, I first recreated the sequencing-read summary (Table 3; recreation of Supplementary Figure S2A). My raw-read totals were similar to those reported in the paper, but my mapped-read counts were higher and more consistent across replicates. These differences likely result from updated trimming and alignment tools that increase mapping efficiency, as well as differences in sequencing depth, library preparation, or quality control thresholds between my dataset and the original study. Despite these numerical differences, the overall interpretation is unchanged: all samples have sufficient depth and mapping quality for downstream ChIP-seq analysis.</font>

&nbsp;&nbsp;&nbsp;<font size="3">Next, I examined replicate similarity by recreating the correlation heatmap (Figure 6; recreation of Supplementary Figure S2B). Consistent with the original authors, I observed strong clustering of the IP replicates and weaker correlation among the INPUT samples. My correlation values were generally higher, indicating greater reproducibility across both IP and INPUT replicates. This likely reflects improved normalization, cleaner library preparation, or differences in signal-matrix generation. Both my heatmap and the original figure support the same conclusion: the IP samples show strong internal consistency, confirming successful ChIP enrichment.</font>

&nbsp;&nbsp;&nbsp;<font size="3">I then compared the reproducible peak set identified between biological replicates (Figure 7; recreation of Supplementary Figure S2C). The original study reported 3,983 Rep1-only peaks, 10,465 Rep2-only peaks, and 3,466 shared peaks, whereas my analysis produced a larger overall peak count and a reproducible overlap of 8,664 peaks. These discrepancies could be attributed to differences in sequencing depth and library complexity, which influence peak-calling sensitivity, and to differences in HOMER versions or filtering parameters that affect the definition of peak overlap. Even minor methodological variations, such as duplicate removal, genome build, or peak-width thresholds, can substantially alter peak numbers.</font>

&nbsp;&nbsp;&nbsp;<font size="3">Across all three recreated outputs (Table 3, Figure 6, and Figure 7), the exact values differ from those in the publication, yet the biological interpretation remains consistent with the original study. My dataset shows high-quality sequencing, strong replicate correlations, and a robust, reproducible RUNX1 peak set. These results support successful ChIP-seq execution and validate my workflow as an accurate re-analysis of the published experiment.</font>



| **Sample Name** | **Biological Replicate** | **Raw Reads** | **Mapped Reads** |
|-----------------|--------------------------|----------------|------------------|
| IP_rep1         | 1                        | 28,578,787     | 27,654,087       |
| IP_rep2         | 2                        | 28,547,028     | 28,062,474       |
| INPUT_rep1      | 1                        | 29,578,193     | 28,463,242       |
| INPUT_rep2      | 2                        | 10,696,005     | 10,031,987       |


**Table 3.Summary of sequencing read counts for all RUNX1 ChIP-seq and INPUT samples (Recreation of Supplemenatry Figure S2A).** This table reports the total number of raw sequencing reads and the number of reads passing QC for each biological replicate. These metrics provide an overview of sequencing depth and data quality prior to alignment and downstream analysis.

<img src="/projectnb/bf528/students/monsivam/project-3-monsivam/Project_3_Figures/Correlation Between ChIP-seq Samples.png" width="700"/>

**Figure 6. RUNX1 Binding Relative to Differentially Expressed Genes (Recreation of Supplemenatry Figure S2B).** This heatmap shows the pairwise Pearson correlation coefficients between all ChIP-seq libraries. IP replicates (IP_rep1 and IP_rep2) cluster together and show strong correlation, indicating high reproducibility of RUNX1 ChIP-seq signal. INPUT samples display moderate correlation with each other and low correlation with IP samples, as expected for background controls.

<img src="/projectnb/bf528/students/monsivam/project-3-monsivam/Project_3_Figures/Overlap of Reproducible Peaks.png" width="700"/>

**Figure 7. Overlap of RUNX1 ChIP-seq peaks (Recreation of Supplemenatry Figure S2C).** RUNX1 ChIP-seq peak calling identified 159,939 peaks in replicate 1 and 23,879 peaks in replicate 2. The intersection of both replicates yielded 8,664 high-confidence peaks, which were considered reproducible and carried forward for downstream annotation and integrative analyses.

#### **Analyze the annotated peaks**
&nbsp;&nbsp;&nbsp;<font size="3">Process enrichment analysis using the Enrichr workflow on genes within ±20 kb of each annotated peak, as shown in Figure 8. The top enriched pathways, visualized as –log10(adjusted p-values), include transcriptional regulation, intracellular signaling, apoptosis, and chromatin modification. These functions are consistent with established roles of RUNX1 in gene expression and cell-state transitions. The results indicate that RUNX1 directly influences transcriptional control and signaling pathways in epithelial contexts. If needed, further analysis can subset genes by genomic location, such as promoter-proximal versus distal peaks, to clarify whether specific processes are associated with TSS binding elements. The enrichment patterns in Figure 7 are consistent with those in the original study, supporting that the RUNX1-associated peaks in my dataset represent biologically meaningful regulatory targets. The overlap between enriched biological processes in my data and those reported in the publication further validates the biological significance of these RUNX1 ChIP-seq peaks.</font>

<img src="/projectnb/bf528/students/monsivam/project-3-monsivam/Project_3_Figures/Gene Ontology enrichment analysis of genes.png" width="700"/>

**Figure 8. Gene Ontology enrichment analysis of genes associated with reproducible RUNX1 ChIP-seq peaks.** This bar plot displays the top GO Biological Process terms enriched among genes located within ±20 kb of reproducible RUNX1-bound regions. Bars represent the –log10(adjusted p-value), where higher values indicate stronger statistical enrichment. The most enriched pathways include transcriptional regulation, intracellular signaling, apoptosis, and chromatin modification, reflecting known functional roles of RUNX1 in gene regulatory control and cellular signaling environments.